<a href="https://colab.research.google.com/github/Blenkzx/LINGUAGENS-DE-PROGRAMA-O/blob/main/Exerc%C3%ADcioPr%C3%A1ticoFolium.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import folium
from folium.plugins import MarkerCluster

# Gerando dados sintéticos de imóveis em Nova Iguaçu e Queimados
np.random.seed(42)
n_imoveis = 45

# Coordenadas base (Aproximadas)
# Nova Iguaçu: -22.756, -43.460
# Queimados: -22.716, -43.555

dados_imoveis = {
    'id_imovel': range(1, n_imoveis + 1),
    'cidade': np.where(np.random.rand(n_imoveis) > 0.4, 'Nova Iguaçu', 'Queimados'),
    'valor_venda': np.random.uniform(150000, 850000, n_imoveis).round(2),
    'tipo': np.random.choice(['Casa', 'Apartamento', 'Terreno'], n_imoveis)
}

df_mapa = pd.DataFrame(dados_imoveis)

# Atribuindo coordenadas com base na cidade adicionando uma pequena dispersão aleatória
def gerar_lat(cidade):
    if cidade == 'Nova Iguaçu':
        return -22.756 + np.random.uniform(-0.03, 0.03)
    return -22.716 + np.random.uniform(-0.02, 0.02)

def gerar_lon(cidade):
    if cidade == 'Nova Iguaçu':
        return -43.460 + np.random.uniform(-0.03, 0.03)
    return -43.555 + np.random.uniform(-0.02, 0.02)

df_mapa['latitude'] = df_mapa['cidade'].apply(gerar_lat)
df_mapa['longitude'] = df_mapa['cidade'].apply(gerar_lon)

### Parte 1: Inicialização e Marcadores Básicos

In [2]:
import folium

# Calcular a coordenada média para centralizar o mapa
lat_media = df_mapa['latitude'].mean()
lon_media = df_mapa['longitude'].mean()

# Criar o mapa base
mapa_basico = folium.Map(location=[lat_media, lon_media], zoom_start=12, control_scale=True)

# Adicionar marcadores para as 5 primeiras linhas
for index, row in df_mapa.head(5).iterrows():
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=f"Tipo: {row['tipo']}<br>Valor: R${row['valor_venda']:,.2f}"
    ).add_to(mapa_basico)

# Exibir o mapa
mapa_basico

### Parte 2: Customização Visual com Marcadores Circulares

In [3]:
# Criar um novo mapa base
mapa_circular = folium.Map(location=[lat_media, lon_media], zoom_start=12, control_scale=True)

# Adicionar CircleMarkers para todos os imóveis
for index, row in df_mapa.iterrows():
    cor_preenchimento = 'blue' if row['cidade'] == 'Nova Iguaçu' else 'orange'
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=8,
        color='black',
        weight=1,
        fill=True,
        fill_color=cor_preenchimento,
        fill_opacity=0.7,
        tooltip="Clique para detalhes",
        popup=f"Tipo: {row['tipo']}<br>Valor: R${row['valor_venda']:,.2f}"
    ).add_to(mapa_circular)

# Exibir o mapa
mapa_circular

### Parte 3: Agrupamento Inteligente (Clustering)

In [4]:
from folium.plugins import MarkerCluster

# Criar um novo mapa base
mapa_cluster = folium.Map(location=[lat_media, lon_media], zoom_start=12, control_scale=True)

# Instanciar MarkerCluster
marker_cluster = MarkerCluster().add_to(mapa_cluster)

# Adicionar todos os imóveis ao cluster com ícones customizados
for index, row in df_mapa.iterrows():
    if row['tipo'] == 'Casa':
        icon_color = 'green'
    elif row['tipo'] == 'Apartamento':
        icon_color = 'blue'
    else:
        icon_color = 'gray'

    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=f"Tipo: {row['tipo']}<br>Valor: R${row['valor_venda']:,.2f}",
        icon=folium.Icon(color=icon_color, icon='home' if row['tipo'] == 'Casa' else 'building' if row['tipo'] == 'Apartamento' else 'tree')
    ).add_to(marker_cluster)

# Salvar o mapa final em HTML
mapa_cluster.save('mapa_imoveis_baixada.html')

# Exibir o mapa
mapa_cluster